In [ ]:
import os, sys, pathlib
_project_root = pathlib.Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatch
from geometry.voronoi import read_capitals

In [ ]:
gdf, capitals_gdf = read_capitals()

fig, ax = plt.subplots(figsize=(12, 8))
gdf.plot(ax=ax, color='lightgrey', edgecolor='black')
capitals_gdf.plot(ax=ax, color='red', markersize=50, marker="*")
ax.set(aspect='equal', xlabel='Longitude', ylabel='Latitude', title='Mainland US State Capitals')
plt.show()


Vonroi
----
***

The Voronoi cell, or Voronoi region, 
${\textstyle R_{k}}$
, associated with the site 
${\textstyle P_{k}}$
 is the set of all points in 
${\textstyle X}$
 whose distance to 
${\textstyle P_{k}}$ 
is not greater than their distance to the other sites 
${\textstyle P_{j}}$, 
where 
${\textstyle j}$ 
is any index different from 
${\textstyle k}$. 
In other words, if 
${\textstyle d(x,\,A)=\inf\{d(x,\,a)\mid a\in A\}}$ 
denotes the distance between the point 
${\textstyle x}$
and the subset 
${\textstyle A}$
, then
$$
{\displaystyle R_{k}=\{x\in X\mid d(x,P_{k})\leq d(x,P_{j})\;{\text{for all}}\;j\neq k\}}
$$

In [ ]:
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d
from shapely.geometry import Polygon, MultiPolygon, Point

def voronoi_finite_polygons_2d(vor: Voronoi, radius=1e6):
	"""
	Reconstruct finite Voronoi polygons in 2D.
	Parameters:
			vor: Voronoi object from scipy.spatial.Voronoi
			radius: Distance to 'close' infinite regions.
	Returns:
			regions: List of regions (each a list of indices of vertices)
			vertices: Array of vertices coordinates.
	"""
	if vor.points.shape[1] != 2:
			raise ValueError("Requires 2D input")
	
	new_regions = []
	new_vertices: list = vor.vertices.tolist()
	
	center = vor.points.mean(axis=0)
	if radius is None:
		radius = vor.points.ptp().max()*2
	
	# Map: point index -> list of ridges (neighbor point, vertex index1, vertex index2)
	all_ridges: dict[str, list] = {}
	for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
		all_ridges.setdefault(p1, []).append((p2, v1, v2))
		all_ridges.setdefault(p2, []).append((p1, v1, v2))
	
	# Reconstruct each Voronoi region
	for p1, region_idx in enumerate(vor.point_region):
		vertices = vor.regions[region_idx]
		if all(v >= 0 for v in vertices):
			# Region is already closed
			new_regions.append(vertices)
			continue
		
		# Region has infinite vertices, so reconstruct it.
		new_region = [v for v in vertices if v >= 0]
		for p2, v1, v2 in all_ridges[p1]:
			if v1 < 0 or v2 < 0:
				# one of the vertices is infinite
				v = v1 if v1 >= 0 else v2
				# Compute the tangent direction between points
				tangent = vor.points[p2] - vor.points[p1]
				tangent /= np.linalg.norm(tangent)
				# Normal direction
				normal = np.array([-tangent[1], tangent[0]])
				# Determine the direction of the ray
				midpoint = vor.points[[p1, p2]].mean(axis=0)
				direction = np.sign(np.dot(midpoint - center, normal)) * normal
				# Compute a far away point for the ray
				far_point = vor.vertices[v] + direction * radius
				new_vertices.append(far_point.tolist())
				new_region.append(len(new_vertices) - 1)
		# Sort region vertices in a counterclockwise order
		vs = np.array([new_vertices[v] for v in new_region])
		c = vs.mean(axis=0)
		angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
		new_region = [v for _, v in sorted(zip(angles, new_region))]
		new_regions.append(new_region)
	
	return new_regions, np.asarray(new_vertices)



In [ ]:
from shapely import get_coordinates
from shapely.plotting import plot_polygon

coords = list(get_coordinates(capitals_gdf.geometry))
coords = np.asarray(coords) # shape (n_points, 2)

tri = Delaunay(coords)
vor = Voronoi(coords)

regions, vertices = voronoi_finite_polygons_2d(vor)
us_boundary = gdf.unary_union
clipped_polygons = []
for region in regions:
	polygon_coords = vertices[region]
	poly = Polygon(polygon_coords)
	poly_clipped = poly.intersection(us_boundary)
	clipped_polygons.append(poly_clipped)

fig, ax = plt.subplots(figsize=(15, 10))
gdf.plot(ax=ax, color='white', edgecolor='white')
capitals_gdf.plot(ax=ax, color='red', markersize=50,  marker="*")


for poly in clipped_polygons:
	if poly.is_empty:
			continue
	plot_polygon(poly, ax=ax, edgecolor='black', add_points=False, color="white")
	# if isinstance(poly, Polygon):
	#     x, y = poly.exterior.xy
	#     ax.fill(x, y, alpha=0.3, edgecolor='black')
	# elif isinstance(poly, MultiPolygon):
	#     plot_polygon(poly)
			# for subpoly in poly:
			#     x, y = subpoly.exterior.xy
			#     ax.fill(x, y, alpha=0.3, edgecolor='black')

ax.set(aspect="equal", xlabel='Longitude', ylabel='Latitude', title='US State Capitals with Bounded Voronoi Diagram')
ax.axis('off')
plt.show()

In [ ]:
verts = vor.vertices
regions = []
for region in vor.regions:
	regions.append(verts[region])


fig, ax = plt.subplots(figsize=(15, 10))
gdf.plot(ax=ax, color='lightgrey', edgecolor='lightgrey')
capitals_gdf.plot(ax=ax, color='red', markersize=50)
for region in regions:
	ax.add_patch(mpatch.Polygon(region, alpha=0.1))
voronoi_plot_2d(vor, ax=ax, show_points=False, show_vertices=False)
ax.set(aspect="equal", xlabel='Longitude', ylabel='Latitude', title='US State Capitals', xlim=(-130, -60), ylim=(25, 50))
plt.show()

In [ ]:
p = tri.points[tri.simplices]
A = p[:,0,:].T
B = p[:,1,:].T
C = p[:,2,:].T
a = A - C
b = B - C

def dot2(u, v):
	return u[0]*v[0] + u[1]*v[1]

def cross2(u, v, w):
	"""u x (v x w)"""
	return dot2(u, w)*v - dot2(u, v)*w

def ncross2(u, v):
	"""|| u x v ||^2"""
	return sq2(u)*sq2(v) - dot2(u, v)**2

def sq2(u):
	return dot2(u, u)

cc = cross2(sq2(a) * b - sq2(b) * a, a, b) / (2*ncross2(a, b)) + C

# Grab the Voronoi edges
vc = cc[:,tri.neighbors]
vc[:,tri.neighbors == -1] = np.nan # edges at infinity, plotting those would need more work...

lines = []
lines.extend(zip(cc.T, vc[:,:,0].T))
lines.extend(zip(cc.T, vc[:,:,1].T))
lines.extend(zip(cc.T, vc[:,:,2].T))

# Plot it
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

lines = LineCollection(lines, edgecolor='k')

# plt.hold(1)

plt.plot(coords[:,0], coords[:,1], '.')
plt.plot(cc[0], cc[1], '*')
plt.gca().add_collection(lines)
plt.axis('equal')
plt.xlim(-200, 100)
plt.ylim(-25, 100)
plt.show()

In [ ]:

def circumcenter(triangle):
    a, b, c = triangle
    print(a)
    ax, ay = coords[a]
    bx, by = coords[b]
    cx, cy = coords[c]
    
    d = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    
    center_x = ((ax**2 + ay**2) * (by - cy) + (bx**2 + by**2) * (cy - ay) + (cx**2 + cy**2) * (ay - by)) / d
    center_y = ((ax**2 + ay**2) * (cx - bx) + (bx**2 + by**2) * (ax - cx) + (cx**2 + cy**2) * (bx - ax)) / d
    
    return center_x, center_y

voronoi_vertices = np.array([circumcenter(coords[simplex]) for simplex in tri.simplices])

In [ ]:
import geonamescache
import pandas as pd
# from matplotlib.patches import Polygon
import geopandas as gpd
import geodatasets
df = pd.read_csv('state_capitals_coords.csv')
ne_path = geodatasets.get_path("ne_110m_admin_0_countries")
world = gpd.read_file(ne_path)
usa = world[world['ADMIN'] == 'United States of America']

gc = geonamescache.GeonamesCache()
us_states = gc.get_us_states()
for state_code, state_info in us_states.items():
	print(f"State code: {state_code}")
	print(state_info)
	break

gdf = gdf.merge(df, left_on='STUSPS',right_on='state')
gdf.plot()